# Model Head Accuracy on ImageNet Val

Runs the full ImageNet validation set through the model's classification head and reports top-1 and top-5 accuracy against ground truth labels.

In [ ]:
import torch
import timm
import timm.data
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from tqdm.auto import tqdm

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
# ── Model selection — uncomment the model you want to use ──────────────────
MODEL_NAME = "vit_large_patch14_clip_224.openai_ft_in1k"   # CLIP ViT-L/14
# MODEL_NAME = "vit_large_patch16_224.augreg_in21k_ft_in1k"  # ViT-L/16 (AugReg)
# MODEL_NAME = "deit3_large_patch16_224.fb_in1k"             # DeiT3-L/16
# MODEL_NAME = "vit_large_patch14_dinov2.lvd142m"           # DINOv2-L/14    (518×518, 37×37 grid)

IMAGENET_ROOT = "/data/imagenet/extracted"   # path with val/ subdir
BATCH_SIZE    = 64
NUM_WORKERS   = 4
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"
# ──────────────────────────────────────────────────────────────────────────────

In [ ]:
# Load model
model = timm.create_model(MODEL_NAME, pretrained=True)
model.eval().to(DEVICE)
for p in model.parameters():
    p.requires_grad_(False)

print(f"Model : {MODEL_NAME}")
print(f"Device: {DEVICE}")

In [ ]:
# Build val dataloader with model-specific transforms
data_config = timm.data.resolve_model_data_config(model)
val_transform = timm.data.create_transform(**data_config, is_training=False)

val_dataset = ImageFolder(f"{IMAGENET_ROOT}/val", transform=val_transform)
val_loader  = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print(f"Val images : {len(val_dataset):,}")
print(f"Val batches: {len(val_loader):,}")

In [ ]:
# Evaluate top-1 and top-5 accuracy
top1_correct = 0
top5_correct = 0
total        = 0

with torch.no_grad():
    for images, labels in tqdm(val_loader, desc="Evaluating"):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        logits = model(images)          # [B, 1000]

        # Top-1
        top1_pred = logits.argmax(dim=-1)
        top1_correct += (top1_pred == labels).sum().item()

        # Top-5
        top5_preds = logits.topk(5, dim=-1).indices  # [B, 5]
        top5_correct += (top5_preds == labels.unsqueeze(1)).any(dim=1).sum().item()

        total += images.size(0)

top1_acc = top1_correct / total * 100
top5_acc = top5_correct / total * 100

print(f"\nResults over {total:,} val images")
print(f"  Top-1 accuracy: {top1_acc:.2f}%")
print(f"  Top-5 accuracy: {top5_acc:.2f}%")